In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pprint import pprint

loader = TextLoader(file_path='../documents/data.txt')

documents = loader.load()
print('这是原文档：')
pprint(documents)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=10,
)

docs = text_splitter.split_documents(documents=documents)

print('这是分割后的文档：')
pprint(docs)

这是原文档：
[Document(metadata={'source': '../documents/data.txt'}, page_content='WebRTC与AI的结合已经催生了许多创新的应用案例，覆盖了多个行业和场景。以下是5个典型的应用案例，展示了WebRTC与AI技术的深度融合：\n一、实时语音翻译与字幕生成\n应用场景：在线会议、跨国远程协作、在线教育。\n技术实现：WebRTC提供实时音视频传输，AI通过语音识别（ASR）将语音转化为文本，并结合自然语言处理（NLP）进行实时翻译和字幕生成。\n案例：Zoom、Microsoft Teams等平台已经集成了实时翻译功能，帮助用户跨越语言障碍。\n\n二、智能视频分析与监控\n应用场景：安防监控、零售分析、智能家居。\n技术实现：WebRTC传输实时视频流，AI通过计算机视觉（CV）技术进行人脸识别、行为分析、物体检测等。\n案例：零售商店利用WebRTC+AI分析顾客行为，优化商品摆放和营销策略；智能家居系统通过视频监控识别异常行为并发出警报。\n\n三、虚拟助手与实时互动\n应用场景：在线客服、虚拟导购、智能教育助手。\n技术实现：WebRTC提供实时音视频交互能力，AI通过语音识别和自然语言处理理解用户需求，并提供智能回复或引导。\n案例：电商平台使用虚拟导购助手，通过WebRTC与用户实时互动，推荐商品并解答问题。\n\n四、远程医疗与健康监测\n应用场景：远程问诊、健康监测、康复指导。\n技术实现：WebRTC实现医生与患者的实时音视频通信，AI通过图像识别分析医疗影像（如X光片），或通过语音识别记录患者病情描述。\n案例：远程医疗平台利用WebRTC+AI技术，为偏远地区患者提供高质量的医疗服务。\n\n五、沉浸式虚拟会议与AI增强\n应用场景：元宇宙会议、虚拟协作空间。\n技术实现：WebRTC提供实时音视频传输，AI通过增强现实（AR）和虚拟现实（VR）技术创建虚拟会议环境，并结合语音识别、情感分析等技术提升会议体验。\n案例：Meta的Horizon Workrooms等平台利用WebRTC+AI技术，打造沉浸式虚拟会议空间。\n')]
这是分割后的文档：
[Document(metadata={'source': '../documents/data.txt'}, page_co

In [ ]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PythonLoader

print([e.value for e in Language])

python_code = PythonLoader(file_path='./test.py').load()

splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.JS,
    chunk_size=64,
    chunk_overlap=0
)

splitter.split_documents(documents=python_code)


In [ ]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

print([e.value for e in Language])

python_code = '''
def my_function(name,job): 
	print("Welcome " + name + ", the " + job);

my_function(name='Harry Potter',job='Wizard')

def for_function():
    for i in range(5):
        print("这个数字是" + str(i))

for_function()
'''

splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=64,
    chunk_overlap=0
)

splitter.create_documents(texts=[python_code])

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
from pprint import pprint
from langchain_openai import OpenAIEmbeddings
import os

embeddings = OpenAIEmbeddings(
    model=os.environ['EMBEDDING_MODEL_NAME'],
    base_url=os.environ['BASE_URL'],
    api_key=os.environ['OPENAI_API_KEY'], # type: ignore
    check_embedding_ctx_length=False
)

# text_splitter = SemanticChunker(embeddings=embeddings)

# loader = PyPDFLoader(file_path='../documents/data.pdf', mode='page')
# rst = loader.load()
# print(rst)

# text_splitter.create_documents(texts=[r.page_content for r in rst], metadatas=[r.metadata for r in rst])
# res = text_splitter.split_documents(documents=rst)

# for r in res:
#     pprint(r.page_content)


In [3]:
from langchain.chains.summarize import load_summarize_chain
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain.text_splitter  import RecursiveCharacterTextSplitter 
import os
import uuid
from pprint import pprint

llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    temperature=0.5,
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"], # type: ignore
)


def preprocess_documents(documents):
    """处理原始文档为文档单元"""
    doc_units = []
    for doc in documents:
        # 按大段落拆分（约500字）
        section_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50 
        )
        sections = section_splitter.split_text(doc.page_content) 
        
        # 创建文档单元 
        for section in sections:
            doc_units.append(Document( 
                page_content=section,
                metadata={"doc_id": str(uuid.uuid4()),  "source": doc.metadata.get("source",  "")}
            ))
    return doc_units 


def generate_summaries(doc_units):
    """为每个文档单元生成摘要"""
    summaries = []
    for unit in doc_units:
        # 使用摘要链生成摘要 
        summary_chain = load_summarize_chain(
            llm,
            chain_type="map_reduce"
        )
        summary = summary_chain.invoke({"input_documents": [unit]})["output_text"] 
        
        # 保存摘要和关联元数据 
        summaries.append({ 
            "summary_text": summary,
            "doc_id": unit.metadata["doc_id"], 
            "source": unit.metadata["source"] 
        })
    return summaries 

doc_units = preprocess_documents(documents=documents)
summaries = generate_summaries(doc_units)
pprint(summaries)

[{'doc_id': '3a029d19-e80b-46ed-9d5b-450e58677acf',
  'source': '../documents/data.txt',
  'summary_text': 'CONCISE SUMMARY: WebRTC enables real-time audio/video '
                  'streaming, while AI provides intelligent processing like '
                  'speech translation/subtitles and computer vision analysis '
                  '(facial recognition, object detection), powering innovative '
                  'applications across industries such as collaboration, '
                  'security, retail, and monitoring.'},
 {'doc_id': 'c47ba1c2-3d9c-47ca-8391-58b2d99113c5',
  'source': '../documents/data.txt',
  'summary_text': "Here's a concise summary of the provided text:\n"
                  '\n'
                  '**WebRTC enables real-time audio/video communication, '
                  'combined with AI for analysis and intelligence, powering '
                  'applications in virtual assistance, remote healthcare, and '
                  'immersive virtual meetings.**'}]


In [5]:
# from langchain_community.vectorstores import Chroma
from langchain_community.vectorstores import FAISS

# 3. 构建摘要索引 
def build_summary_index(summaries):
    """创建摘要向量索引"""
    summary_texts = [s["summary_text"] for s in summaries]
    summary_metadatas = [{"doc_id": s["doc_id"], "source": s["source"]} for s in summaries]

    print('这是摘要文本：')
    print(summary_metadatas)
    
    # 创建Chroma索引 [8]()
    # summary_index = Chroma.from_texts( 
    #     texts=summary_texts,
    #     embedding=embeddings,
    #     metadatas=summary_metadatas 
    # )

    summary_index = FAISS.from_texts(
        texts=summary_texts,
        embedding=embeddings,
        metadatas=summary_metadatas
    )

    summary_index.save_local(folder_path="../db/python/summary")

    return summary_index 

# 4. 构建内容块索引 
def build_chunk_index(doc_units):
    """创建内容块向量库"""
    all_chunks = []
    
    for unit in doc_units:
        # 细粒度拆分（约100字） [2]()
        chunk_splitter = RecursiveCharacterTextSplitter(
            chunk_size=100,
            chunk_overlap=20 
        )
        chunks = chunk_splitter.split_text(unit.page_content) 
        
        # 添加块元数据 
        for chunk in chunks:
            all_chunks.append(Document( 
                page_content=chunk,
                metadata={
                    "doc_id": unit.metadata["doc_id"], 
                    "chunk_id": str(uuid.uuid4()), 
                    "source": unit.metadata["source"] 
                }
            ))
    
    # 创建Chroma索引 
    chunk_index = FAISS.from_documents( 
        documents=all_chunks,
        embedding=embeddings 
    )
    chunk_index.save_local(folder_path="../db/python/chunk")
    return chunk_index 

# 5. 分层检索 
def hierarchical_retrieval(query, summary_index, chunk_index, top_k=3):
    """执行分层检索"""
    # 第一阶段：摘要检索 [1]()
    relevant_summaries = summary_index.similarity_search(query,  k=top_k)
    doc_ids = [s.metadata["doc_id"] for s in relevant_summaries]
    
    # 第二阶段：块检索（基于关联文档）
    relevant_chunks = chunk_index.max_marginal_relevance_search( 
        query,
        k=top_k,
        filter={"doc_id": {"$in": doc_ids}}  # 元数据过滤 [6]()
    )
    
    return relevant_chunks 

# 6. 生成答案 
def generate_answer(query, chunks):
    """使用LLM生成最终答案"""
    context = "\n\n".join([c.page_content for c in chunks])
    prompt = f"""
    基于以下上下文回答问题：
    {context}
    
    问题：{query}
    答案：
    """
    return llm.invoke(prompt)


summary_index = build_summary_index(summaries)

chunk_index = build_chunk_index(doc_units)

query = "在线客服是怎么实现的？"

relevant_chunks = hierarchical_retrieval(query, summary_index, chunk_index)

answer = generate_answer(query, relevant_chunks)

print("最终答案：")
print(answer)

这是摘要文本：
[{'doc_id': '3a029d19-e80b-46ed-9d5b-450e58677acf', 'source': '../documents/data.txt'}, {'doc_id': 'c47ba1c2-3d9c-47ca-8391-58b2d99113c5', 'source': '../documents/data.txt'}]
最终答案：
content='基于上下文提供的信息，在线客服的实现主要依赖于以下技术组合和流程：\n\n### 核心技术实现：\n1. **实时音视频交互（WebRTC）**  \n   - **作用**：提供低延迟的实时通信能力，支持用户与客服进行语音、视频或屏幕共享互动。  \n   - **应用场景**：当用户需要面对面咨询或复杂问题演示时（如技术故障排查），通过WebRTC建立点对点连接，确保流畅的音视频传输。\n\n2. **AI语音识别与自然语言处理（NLP）**  \n   - **语音识别**：将用户的语音输入实时转换为文本，供AI系统分析。  \n   - **自然语言处理**：  \n     - **意图识别**：理解用户问题的核心需求（如“查询订单”“投诉售后”）。  \n     - **语义分析**：解析上下文，识别关键词、情绪（如焦虑、急切）及隐含需求。  \n     - **智能回复生成**：基于预设知识库或大语言模型（LLM），自动生成精准、自然的回复或操作建议（如引导用户自助操作、转接人工客服）。\n\n3. **多模态交互支持**  \n   - 除语音外，系统可同时处理文字输入（如聊天框）、图片上传（如故障截图）等，通过AI统一分析并整合回复。\n\n### 完整工作流程：\n1. **用户接入**  \n   - 用户通过网页/App发起客服请求，系统通过WebRTC建立实时通信通道（音视频/文字）。\n\n2. **AI实时处理**  \n   - **语音/文字输入**：用户语音由WebRTC传输至AI引擎，语音识别模块转为文本；文字输入直接进入NLP模块。  \n   - **需求分析**：NLP模型解析文本，识别意图、提取关键信息（如订单号、问题描述）。  \n   - **智能决策**：  \n     - **简单问题**：AI直接从知识库匹配答案并回

In [ ]:
import chromadb
chroma_client = chromadb.Client()

collection = chroma_client.create_collection(name="my_collection")

collection.add(
    documents=["This is a document about engineer", "This is a document about steak"],
    metadatas=[{"source": "doc1"}, {"source": "doc2"}],
    ids=["id1", "id2"]
)

results = collection.query(
    query_texts=["Which food is the best?"],
    n_results=2
)

print(results)